# Patch 단위 pair-separability eval (정량)

PCA(반사 축, 둔감) 대신 **혼동 물질 pair 가 실제로 갈리는지**를 숫자로 측정.
체크포인트 **1개**를 넣고 실행 → 혼동 pair별:
- **eval centroid cosine** — 낮을수록 분리 (train `pair_sim_max` 의 held-out 판박이)
- **AUC** (cls_a vs cls_b linear probe, **이미지 단위 split** — patch leakage 방지)
- (하단) 최혼동 pair 의 **LDA 1D 히스토그램** (PCA 대신 판별축)

**사용법:** CKPT 를 hires 이전(stage2)으로 놓고 한 번, 이후(stage3_1k)로 놓고 한 번 →
출력 표 두 개를 비교. cosine↓ / AUC↑ 면 개선.

정규화는 학습과 동일 instance_norm(PerImageNormalize).

## 0. 설정 (여기만 수정 — CKPT 바꿔 before/after 두 번 실행)

In [ ]:
CONFIG_FILE = '../dino_v3/dinov3/configs/train/weaksup/stage3_hires_adapt.yaml'
CKPT        = '/서버/stage2/eval/xxx/teacher_checkpoint.pth'   # ← before/after 여기만 교체
DATA_ROOT   = '/서버/labeled_eval'    # images/ + masks/ (held-out 권장). 단일 task 폴더.
IMAGE_SIZE  = 1024                    # 1k 로 평가 (2k 도 바꿔서 비교 가능)
N_BLOCKS    = 1
PATCH_SIZE  = 16
PURITY      = 0.8                     # boundary patch 제외 (학습과 동일)
MIN_PATCHES = 10                      # class 당 최소 patch 수
BG_CLASS    = 0
MAX_IMAGES  = 60
N_SPLITS    = 5                       # 이미지 단위 split 반복
TEST_FRAC   = 0.4
SEED        = 0

## 1. import + 헬퍼

In [ ]:
import os, sys
from itertools import combinations
import numpy as np
import torch
from PIL import Image

_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_DINOV3 = os.path.join(_ROOT, 'dino_v3')
for _p in (_DINOV3, _ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import dinov3.distributed as distributed
from dinov3.eval.setup import setup_and_build_model
from dinov3.eval.em_aug import build_em_eval_transform
from dinov3.data.labeled import LabeledEMDataset
from dinov3.data.labeled.patch_label import mask_to_patch_labels

@torch.inference_mode()
def extract_patch_map(model, x, n_blocks, autocast_dtype):
    with torch.autocast('cuda', dtype=autocast_dtype):
        outs = model.get_intermediate_layers(x, n=n_blocks, reshape=False, return_class_token=True, norm=True)
    patch = outs[-1][0][0].float().cpu().numpy()   # [N,C]
    n = patch.shape[0]; g = int(round(n ** 0.5))
    assert g * g == n, f'grid 아님 N={n}'
    return patch, g

def l2norm(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

print('헬퍼 준비 완료')

## 2. 모델 로드 + feature/label 수집 (이미지별)

In [ ]:
if not distributed.is_enabled():
    distributed.enable(overwrite=True)

model, ctx = setup_and_build_model(
    config_file=CONFIG_FILE, pretrained_weights=CKPT, output_dir='./_paireval_tmp')
model.cuda().eval()
autocast_dtype = ctx['autocast_dtype']
eval_tf = build_em_eval_transform(IMAGE_SIZE)
print('모델 로드:', CKPT)

ds = LabeledEMDataset(DATA_ROOT, min_nonbg_classes=0)
n_img = min(len(ds), MAX_IMAGES)

# per_img: list of (feats[N,C] L2, labels[N]) — 이미지 단위 split 용
per_img = []
for i in range(n_img):
    image, mask, meta = ds[i]
    x = eval_tf(image).unsqueeze(0).cuda(non_blocking=True)
    f, g = extract_patch_map(model, x, N_BLOCKS, autocast_dtype)
    marr = mask.numpy() if isinstance(mask, torch.Tensor) else np.asarray(mask)
    if marr.ndim == 3:
        marr = marr[..., 0]
    m_res = np.array(Image.fromarray(marr.astype(np.int32)).resize(
        (g * PATCH_SIZE, g * PATCH_SIZE), Image.NEAREST))
    lbl = mask_to_patch_labels(torch.from_numpy(m_res.astype(np.int64)),
                               PATCH_SIZE, PURITY, -1).numpy()
    per_img.append((l2norm(f), lbl))

all_labels = np.concatenate([l for _, l in per_img])
classes = sorted(int(c) for c in np.unique(all_labels)
                 if c >= 0 and c != BG_CLASS and (all_labels == c).sum() >= MIN_PATCHES)
print(f'{n_img} 장, non-bg class {classes}')

## 3. pair별 eval cosine + 이미지단위 AUC

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# class 별 global centroid (L2)
cent = {}
for c in classes:
    feats_c = np.concatenate([f[l == c] for f, l in per_img if (l == c).any()])
    cent[c] = l2norm(feats_c.mean(0))

def pair_auc(a, b):
    # 이미지 단위 split: patch 를 이미지 id 와 함께 모아 이미지 기준으로 train/test 분리
    X, y, img = [], [], []
    for idx, (f, l) in enumerate(per_img):
        for cls, yy in ((a, 0), (b, 1)):
            m = (l == cls)
            if m.any():
                X.append(f[m]); y.append(np.full(m.sum(), yy)); img.append(np.full(m.sum(), idx))
    if not X:
        return np.nan
    X = np.concatenate(X); y = np.concatenate(y); img = np.concatenate(img)
    imgs_with = np.unique(img)
    rng = np.random.default_rng(SEED)
    aucs = []
    for _ in range(N_SPLITS):
        te_imgs = set(rng.choice(imgs_with, max(1, int(len(imgs_with) * TEST_FRAC)), replace=False).tolist())
        te = np.isin(img, list(te_imgs)); tr = ~te
        if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
            continue
        clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
        clf.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs)) if aucs else np.nan

rows = []
for a, b in combinations(classes, 2):
    cos = float(cent[a] @ cent[b])
    auc = pair_auc(a, b)
    rows.append((a, b, cos, auc))
rows.sort(key=lambda r: (-r[2], r[3]))   # 혼동 큰 순 (cosine 높고 AUC 낮은)

print(f'{"pair":<10}{"eval_cos":>10}{"AUC":>8}   (cos↓/AUC↑ = 분리 잘됨)')
print('-' * 40)
for a, b, cos, auc in rows:
    print(f'{str((a,b)):<10}{cos:>10.3f}{auc:>8.3f}')
print('\n★ 최혼동 pair:', (rows[0][0], rows[0][1]),
      f'cos={rows[0][2]:.3f} AUC={rows[0][3]:.3f}')

## 4. 최혼동 pair — LDA 1D 히스토그램 (PCA 대신 판별축)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

a, b = rows[0][0], rows[0][1]
Xa = np.concatenate([f[l == a] for f, l in per_img if (l == a).any()])
Xb = np.concatenate([f[l == b] for f, l in per_img if (l == b).any()])
X = np.concatenate([Xa, Xb]); y = np.array([0] * len(Xa) + [1] * len(Xb))
z = LinearDiscriminantAnalysis(n_components=1).fit_transform(X, y).ravel()

plt.figure(figsize=(7, 4))
plt.hist(z[y == 0], bins=60, alpha=0.6, label=f'cls {a}', density=True)
plt.hist(z[y == 1], bins=60, alpha=0.6, label=f'cls {b}', density=True)
plt.title(f'LDA 판별축 투영 — pair ({a},{b})  [겹칠수록 미분리]')
plt.legend(); plt.tight_layout(); plt.show()